In [0]:
%pip install catboost lightgbm optuna

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # one-liner charts, high-level
import plotly.graph_objects as go # full control chart

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.base import clone

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier, Pool

import optuna

import typing

import warnings

In [0]:
TARGET = "Will_Buy_EV"
N_SPLITS = 5

cat_columns = [
    "Gender",
    "City_Type",
    "Current_Car_Type"
    # "Home_Charging_Possible",
    # "Subsidy_Available",
    # "Range_Anxiety_Level",
]
num_columns = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

yes_no_columns = ["Home_Charging_Possible", "Subsidy_Available"]
ordinal_columns = ["Range_Anxiety_Level"]

In [0]:
def load_data():
    df_train = pd.read_csv("train.csv")
    df_test = pd.read_csv("test.csv")

    df_test["source"] = "test"
    df_train["source"] = "train"

    df = pd.concat([df_train, df_test], ignore_index=True)

    df[TARGET] = df[TARGET].str.lower().map({"no": 0, "yes": 1})
    return df

In [0]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train = df[df['source'] == 'train'].drop(columns=["id", "source"] + additional_drop_columns)
    submission_df = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train.drop(columns=[TARGET])
    y_train = train[TARGET]

    return X_train, y_train, submission_df

In [0]:
df = load_data()

In [0]:
df.head()

# EDA

## Info

In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Adversarial Validation

See if the distributions of train and test sets match

In [0]:
X_adv = df.drop(columns=[TARGET, "source", "id"])
y_adv = df["source"].map({"train": 0, "test": 1})

for c in cat_columns:
    X_adv[c] = X_adv[c].astype("category")

cv_adv = StratifiedKFold(5, shuffle=True, random_state=42)
oof_adv = cross_val_predict(
    LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
    X_adv,
    y_adv,
    cv=cv_adv,
    method="predict_proba",
)[:, 1]
print(roc_auc_score(y_adv, oof_adv))

## EDA on Features

In [0]:
# Income to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Annual_Income_USD',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=30
)

In [0]:
df[df['Annual_Income_USD'].between(31_004, 41_970)]['Will_Buy_EV'].unique()

In [0]:
df[df['source'] == 'train'].groupby('Annual_Income_USD')['Will_Buy_EV'].agg(rate='mean', n='size').sort_index()

In [0]:
df.groupby('Annual_Income_USD')['Will_Buy_EV'].value_counts(normalize=True).reset_index().query('proportion in [0, 1]')

In [0]:
# Income to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Annual_Income_USD',
    hue=TARGET,
    # multiple='fill',
    kde=True,
    bins=30
)

In [0]:
# Daily Commute to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Daily_Commute_km',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=18
)

### Corelations

In [0]:
z = df[df['source']=='train'].copy()

z[['Home_Charging_Possible', 'Subsidy_Available']] = z[['Home_Charging_Possible', 'Subsidy_Available']].apply(lambda col: col.str.lower().map({'yes': 1, 'no': 0}))

z['Range_Anxiety_Level'] = z['Range_Anxiety_Level'].map({'Low':0, 'Medium':1, 'High': 2})

z = pd.concat([z, pd.get_dummies(df['Gender'], prefix='Gender').astype(int)], axis=1)
z.drop(columns=['Gender'], inplace=True)

z = pd.concat([z, pd.get_dummies(df['Current_Car_Type'], prefix='Current_Car_Type').astype(int)], axis=1)
z.drop(columns=['Current_Car_Type'], inplace=True)

z = pd.concat([z, pd.get_dummies(df['City_Type'], prefix='City_Type').astype(int)], axis=1)
z.drop(columns=['City_Type'], inplace=True)

In [0]:
z_corr = z.corr(numeric_only=True)
corr_sort_cols = z_corr['Will_Buy_EV'].abs().sort_values(ascending=False).index.to_list()
z_corr_sorted = z_corr.loc[corr_sort_cols, corr_sort_cols]
z_corr_mask = np.triu(np.ones_like(z_corr_sorted, dtype=bool), k=1)
z_corr_sorted[z_corr_mask] = np.nan

px.imshow(
    z_corr_sorted,
    aspect=True,
    color_continuous_scale="RdBu_r",
    text_auto='.2f'
).update_traces(hoverongaps=True).update_layout(width=1200, height=500)

### Environmental Concern Level

In [0]:
px.histogram(
    df,
    x='Environmental_Concern_Level',
    color='Will_Buy_EV',
    barmode='overlay',
    marginal='box',
    opacity=0.6
)

In [0]:
px.histogram(
    df,
    x='Range_Anxiety_Level',
    color='Will_Buy_EV',
    barmode='overlay',
    # marginal='box',
    opacity=0.6
)

In [0]:
df

In [0]:
fig = px.histogram(
    df,
    x='Annual_Income_USD',
    color='Will_Buy_EV',
    barmode='overlay',
    marginal='box',
    opacity=0.6,
    nbins=10
)

displayHTML(fig.to_html(include_plotlyjs='cdn'))

# Feature Engineering

In [0]:
df.head()

In [0]:
def feature_engineering(df):
    for c in df[yes_no_columns]:
        df[c] = df[c].str.lower().map({"no": 0, "yes": 1})


    df["Range_Anxiety_Level"] = (
        df["Range_Anxiety_Level"].str.lower().map({"low": 0, "medium": 1, "high": 2})
    )

    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work'] + df['Home_Charging_Possible']

    # df['Km_Per_Station'] = df['Daily_Commute_km'] / df['Total_Charging_Stations'].replace(0, np.nan)
    # df['No_Charging'] = (df['Total_Charging_Stations'] == 0).astype('int8')

    df['Realistic_Concern'] = df['Environmental_Concern_Level'] - df['Range_Anxiety_Level']
    
    df["Anxiety_Environmental"] = (
    df["Range_Anxiety_Level"].astype(str)
    + "_"
    + df["Environmental_Concern_Level"].astype(str))

    df['Anxiety_Commute_Ratio'] = np.where(
        df['Daily_Commute_km'] == 0,
        0,
        df['Range_Anxiety_Level'] / df['Daily_Commute_km']
    )

    df["High_Income_EV_Potential"] = (df["Annual_Income_USD"] >= 170537.0).astype('int8')
    df['income_per_car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)

    # # The Dead Zone (0% buy rate region)
    # df['is_dead_zone'] = ((df['Annual_Income_USD'] >= 38000.0) & (df['Annual_Income_USD'] <= 42000.0)).astype('int8')

    # Subsidy combinations

    df['subsidy_income'] = df['Annual_Income_USD'] * df['Subsidy_Available']
    df['subsidy_concern'] = df['Environmental_Concern_Level'] * df['Subsidy_Available']
    df['home_charging_subsidy'] = df['Home_Charging_Possible'] * df['Subsidy_Available']

    num_cols = [
        'Age', 'Annual_Income_USD', 'Daily_Commute_km',
        'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
        'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
    ]
    for c in num_cols:
        for k in range(-4, 4):
            df[f'{c}_digit{k}'] = (df[c].fillna(0) // (10.0 ** k) % 10).astype('int8')
    

    return df

In [0]:
df['Home_Charging_Possible'].value_counts()

In [0]:
df[df["Annual_Income_USD"] >= 170537.0]['Will_Buy_EV'].value_counts()

# Baseline model (Logistic Regression)

In [0]:
df.head()

In [0]:
X, y, submission_set = data_splitter(df)

In [0]:
log_reg_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('scaler', StandardScaler(), num_columns),
    ],
    remainder="drop"
)

log_reg_pipeline = Pipeline([
    ('preprocessor', log_reg_preprocessor),
    ('logreg', LogisticRegression(max_iter=1000))
])

In [0]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_set))
folds_train_results, folds_validation_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    log_reg_pipeline.fit(X_train, y_train)
    oof_pred = log_reg_pipeline.predict_proba(X_val)[:,1]
    train_pred = log_reg_pipeline.predict_proba(X_train)[:,1]
    
    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    folds_train_results.append(train_auc)
    folds_validation_results.append(oof_auc)
    submission_results += log_reg_pipeline.predict_proba(submission_set)[:,1] / N_SPLITS

    print(f"=======FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc} | TEST AUC: {oof_auc} | DELTA {oof_auc - train_auc}")

tr, va = np.array(folds_train_results), np.array(folds_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

## Baseline ROC AUC

TRAIN AVG AUC: 0.93811 +/- 0.00020
VAL AVG AUC: 0.93810 +/- 0.00081

OOF AUC: 0.93809

# XGBoost

In [0]:
df = load_data()
df_ = feature_engineering(df)
X, y, submission_set = data_splitter(df_)

xgb_params = {
    "n_estimators": 1095,
    "max_depth": 8,
    "learning_rate": 0.06757009447016467,
    "subsample": 0.8526064253616809,
    "colsample_bytree": 0.713964646305348,
    "min_child_weight": 39,
    "reg_alpha": 7.741721190554106e-05,
    "reg_lambda": 0.009405232270558986,
    "gamma": 0.6884035558163024,
}

xgb_model = XGBClassifier(**xgb_params, random_state=42, n_jobs=1)

xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
    ],
    remainder="passthrough",
)

xgb_pipeline = Pipeline([("preprocessor", xgb_preprocessor), ("xgb", xgb_model)])

In [0]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_set))
folds_train_results, folds_validation_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    xgb_pipeline.fit(X_train, y_train)
    oof_pred = xgb_pipeline.predict_proba(X_val)[:,1]
    train_pred = xgb_pipeline.predict_proba(X_train)[:,1]
    
    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    folds_train_results.append(train_auc)
    folds_validation_results.append(oof_auc)
    submission_results += xgb_pipeline.predict_proba(submission_set)[:,1] / N_SPLITS

    print(f"=======FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc} | TEST AUC: {oof_auc} | DELTA {oof_auc - train_auc}")

tr, va = np.array(folds_train_results), np.array(folds_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

# LightGBM

In [0]:
df = load_data()
df = feature_engineering(df)
df[['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental']] = df[['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental']].astype('category')
X, y, submission_df = data_splitter(df)

In [0]:
submission_df.info()

In [0]:
def objective_lgb(trial):
    max_depth = trial.suggest_int("max_depth", 4, 12)
    params = {
        "n_estimators": 3000,          # fixed; early stopping decides the real count
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, min(255, 2 ** max_depth)),
        "max_depth": max_depth,
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    oof_local = np.zeros(len(X))
    iters = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]


            model = LGBMClassifier(**params)
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                categorical_feature=['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental'],
                eval_metric="auc",
                callbacks=[early_stopping(100, verbose=False)],
            )

            oof_local[val_idx] = model.predict_proba(X_val)[:, 1]
            iters.append(model.best_iteration_)

            # let Optuna kill hopeless trials early
            trial.report(roc_auc_score(y_val, oof_local[val_idx]), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()


    trial.set_user_attr("best_iters", iters)
    return roc_auc_score(y, oof_local)
# Re-split X and y to include the newly engineered features
# X, y, submission_df = data_spliter(df)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10 ,n_warmup_steps=2),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective_lgb, n_trials=100, show_progress_bar=True)

print(study.best_value)
print(study.best_params)
# how much of the top is noise?
print(study.trials_dataframe().sort_values("value", ascending=False)["value"].head(15).to_string())

In [0]:
print(study.best_value)
print(study.best_params)

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id"])

In [0]:
# Columns in submission_df but not in X
set(X_sub_raw.columns) - set(X.columns)

In [0]:
lgb_test_params = {'n_estimators': 3000, 'max_depth': 4, 'learning_rate': 0.03755714668175416, 'num_leaves': 16, 'min_child_samples': 118, 'subsample': 0.8960672366371799, 'subsample_freq': 5, 'colsample_bytree': 0.5117178113431882, 'reg_alpha': 5.7764356817014716e-08, 'reg_lambda': 1.5042296595462026e-08}

lgb_optuna_params_1 = {'n_estimators': 3000, 'max_depth': 4, 'learning_rate': 0.03597271512159211, 'num_leaves': 16, 'min_child_samples': 193, 'subsample': 0.9146558054022962, 'subsample_freq': 1, 'colsample_bytree': 0.5675296747189175, 'reg_alpha': 6.202845357521301e-08, 'reg_lambda': 0.00015263204457498}

In [0]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]


    model = LGBMClassifier(**lgb_optuna_params_1, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental'],
        eval_metric="auc",
        callbacks=[early_stopping(100, verbose=False)],
    )

    oof_pred = model.predict_proba(X_val)[:, 1]
    train_pred = model.predict_proba(X_train)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

# Catboost

In [0]:
df = load_data()

In [0]:
df.head()

In [0]:
df = load_data()
df = feature_engineering(df)
df[['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental', 'Range_Anxiety_Level']] = df[['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental', 'Range_Anxiety_Level']].astype('str')
catboost_categories = ['Gender','City_Type','Current_Car_Type', 'Anxiety_Environmental', 'Range_Anxiety_Level']
X, y, submission_df = data_splitter(df)

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_df))
fold_train_results, fold_validation_results = [], []
best_iters = []

X_sub_raw = submission_df.drop(columns=["id"])
sub_pool = Pool(X_sub_raw[X.columns], cat_features=catboost_categories)

catboost_test_params = {
    "iterations": 5000,
    "learning_rate": 0.02,
    "depth": 7,
    "l2_leaf_reg": 13,
    "random_strength": 3.4165471601740314,
#     "min_data_in_leaf": 100,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.9557182534037572,
    "border_count":250,
    "one_hot_max_size":2,
    "use_best_model": True,
    "allow_writing_files": False,
    "random_state": 42,
    "verbose": 0,
}

In [0]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
  with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_train = y.iloc[tr_idx]
    y_val =  y.iloc[val_idx]

    train_pool = Pool(X.iloc[tr_idx], y_train, cat_features=catboost_categories)
    valid_pool = Pool(X.iloc[val_idx], y_val, cat_features=catboost_categories)


    model = CatBoostClassifier(
        **catboost_test_params,
        eval_metric='AUC',
        early_stopping_rounds=200,
    )
    model.fit(
        train_pool,
        eval_set=valid_pool,
        verbose=200
    )

    oof_pred = model.predict_proba(valid_pool)[:, 1]
    train_pred = model.predict_proba(train_pool)[:, 1]

    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    fold_train_results.append(train_auc)
    fold_validation_results.append(oof_auc)
    best_iters.append(model.best_iteration_)

    submission_results += model.predict_proba(X_sub_raw)[:, 1] / N_SPLITS

    print(f"======= FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc:.5f} | VAL AUC: {oof_auc:.5f} | "
          f"DELTA: {oof_auc - train_auc:+.5f} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(fold_train_results), np.array(fold_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iters):.0f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

In [0]:
======= FOLD: 0 =======
TRAIN AUC: 0.94494 | VAL AUC: 0.94096 | DELTA: -0.00398 | BEST ITER: 1641
======= FOLD: 1 =======
TRAIN AUC: 0.94494 | VAL AUC: 0.94187 | DELTA: -0.00307 | BEST ITER: 1724
======= FOLD: 2 =======
TRAIN AUC: 0.94448 | VAL AUC: 0.94321 | DELTA: -0.00127 | BEST ITER: 1592
======= FOLD: 3 =======
TRAIN AUC: 0.94460 | VAL AUC: 0.94262 | DELTA: -0.00198 | BEST ITER: 1596
======= FOLD: 4 =======
TRAIN AUC: 0.94471 | VAL AUC: 0.94214 | DELTA: -0.00257 | BEST ITER: 1645

====== CROSS VALIDATION ENDED ======
TRAIN AVG AUC: 0.94466 +/- 0.00026 | VAL AVG AUC: 0.94215 +/- 0.00076
MEAN BEST ITER: 1565

OOF AUC: 0.94215

# Submission

In [0]:
submission_df[TARGET] = submission_results
lgb_prediction = submission_df[['id', TARGET]]
# lgb_prediction.to_csv('lgb_prediction_v6.csv', index=False)

In [0]:
submit_df = submission_set.copy()
submit_df[TARGET] = submission_results

In [0]:
submit_df[['id', TARGET]].to_csv('xgb_submission_v1.csv', index=False)

In [0]:
submit_df[['id', TARGET]]